In [12]:
# Data preparation

In [2]:
pip install xgboost

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simpleNote: you may need to restart the kernel to use updated packages.

     ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
     ---------------------------------------- 0.2/72.0 MB 4.5 MB/s eta 0:00:17
     ---------------------------------------- 0.3/72.0 MB 3.9 MB/s eta 0:00:19
     ---------------------------------------- 0.6/72.0 MB 4.8 MB/s eta 0:00:16
      --------------------------------------- 1.0/72.0 MB 5.8 MB/s eta 0:00:13
      --------------------------------------- 1.4/72.0 MB 6.1 MB/s eta 0:00:12
     - -------------------------------------- 2.0/72.0 MB 7.4 MB/s eta 0:00:10
     - -------------------------------------- 2.5/72.0 MB 7.9 MB/s eta 0:00:09
     - -------------------------------------- 2.7/72.0 MB 7.6 MB/s eta 0:00:10
     - -------------------------------------- 3.0/72.0 MB 7.9 MB/s eta 0:00:09
     - -------------------------------------- 3.0/72.0 MB 7.9 MB/s eta 0:00:09
     - ----

In [4]:
!pip install mlxtend

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     - -------------------------------------- 0.1/1.4 MB 1.6 MB/s eta 0:00:01
     ---- ----------------------------------- 0.2/1.4 MB 1.5 MB/s eta 0:00:01
     --------- ------------------------------ 0.3/1.4 MB 2.4 MB/s eta 0:00:01
     ---------------- ----------------------- 0.6/1.4 MB 3.2 MB/s eta 0:00:01
     ---------------------------- ----------- 1.0/1.4 MB 4.4 MB/s eta 0:00:01
     ---------------------------------------  1.4/1.4 MB 5.0 MB/s eta 0:00:01
     ---------------------------------------- 1.4/1.4 MB 4.5 MB/s eta 0:00:00


In [7]:
!pip install shap

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from mlxtend.frequent_patterns import apriori, association_rules
import shap
import joblib

path='E:/hengseng/6004 data mining/group assignment/new project/'
df = pd.read_csv(path+'survey.csv')

print("The first five rows of data:")
print(df.head())

print("\nALL field names:")
print(df.columns.tolist())

The first five rows of data:
             Timestamp  Age  Gender         Country state self_employed  \
0  2014-08-27 11:29:31   37  Female   United States    IL           NaN   
1  2014-08-27 11:29:37   44       M   United States    IN           NaN   
2  2014-08-27 11:29:44   32    Male          Canada   NaN           NaN   
3  2014-08-27 11:29:46   31    Male  United Kingdom   NaN           NaN   
4  2014-08-27 11:30:22   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often            6-25  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely            6-25  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

                leave mental_health_consequence phys_health_consequence  \
0       Somewhat easy                        No 

In [ ]:
# Data cleaning

In [15]:
# 1.  Find Data missing 
print("Number of missing values in each column:")
print(df.isnull().sum())

# 2. Deal data missing value（save useful values）
# important objects：Age, Gender, family_history, remote_work, benefits, care_options, treatment, work_interfere
key_cols = ["Age", "Gender", "family_history", "remote_work", "benefits", "care_options", "treatment", "work_interfere"]

# Delete usefless data values（简化分析）
df_clean = df[key_cols].copy()

# Deal numeric missing values（Age--mode，避免异常值影响）
df_clean['Age']= df_clean["Age"].fillna(df_clean["Age"].median())

# Deal Categorical type missing values（“Unknown” fill,不浪费样本）
for col in key_cols[1:]: 
    df_clean[col]=df_clean[col].fillna("Unknown")

# 3. Deal Outlier（age<=18 and age>=65,because it is very unreasonable）
df_clean = df_clean[(df_clean["Age"] >= 18) & (df_clean["Age"] <= 65)]

# 4. Simplify "gender" expression（Uniform gendered expression）
df_clean["Gender"] = df_clean["Gender"].apply(lambda x: 
    "Male" if x.lower() in ["male", "m", "man"] else
    "Female" if x.lower() in ["female", "f", "woman"] else
    "Other"
)


print("\nShape of cleaned data (rows×columns)：", df_clean.shape)
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

# save
df_clean.to_csv("cleaned_survey_data.csv", index=False)
print("\nCleaned data has been saved as cleaned_survey_data.csv")

Number of missing values in each column:
Timestamp                       0
Age                             0
Gender                          0
Country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1095
dtype: int64

Shape of cleaned data (rows